# Music Generation with REMI and Transformer
This notebook implements a Decoder-only Transformer trained on MAESTRO v2. Includes Data Preprocessing, AMP Training, and CosineAnnealingLR.

In [ ]:
!pip install miditoolkit
!apt-get update && apt-get install -y fluidsynth fluid-soundfont-gm ffmpeg

## 1. REMI Tokenizer
This module defines the Vocabulary and the Tokenizer mapping musical events to integers.

In [ ]:
import miditoolkit
import math
class Vocabulary:
    def __init__(self):
        self.token2id = {}
        self.id2token = {}
        self.pad_token = '<PAD>'
        self.bos_token = '<BOS>'
        self.eos_token = '<EOS>'
        self.add_token(self.pad_token)
        self.add_token(self.bos_token)
        self.add_token(self.eos_token)
    def add_token(self, token):
        if token not in self.token2id:
            idx = len(self.token2id)
            self.token2id[token] = idx
            self.id2token[idx] = token
    def encode(self, tokens):
        return [self.token2id.get(t, self.token2id[self.pad_token]) for t in tokens]
    def decode(self, ids):
        return [self.id2token.get(i, self.pad_token) for i in ids]
    def __len__(self):
        return len(self.token2id)
    @property
    def pad_id(self):
        return self.token2id[self.pad_token]
    @property
    def bos_id(self):
        return self.token2id[self.bos_token]
    @property
    def eos_id(self):
        return self.token2id[self.eos_token]
class REMITokenizer:
    def __init__(self):
        self.vocab = Vocabulary()
        self.pitch_range = range(21, 109)
        self.velocity_bins = 32
        self.duration_bins = 64
        self.position_resolution = 16
        self._build_vocab()
    def _build_vocab(self):
        self.vocab.add_token("Bar")
        for i in range(self.position_resolution):
            self.vocab.add_token(f"Position_{i}")
        for p in self.pitch_range:
            self.vocab.add_token(f"Pitch_{p}")
        for v in range(self.velocity_bins):
            self.vocab.add_token(f"Velocity_{v}")
        for d in range(1, self.duration_bins + 1):
            self.vocab.add_token(f"Duration_{d}")
    def _quantize_velocity(self, velocity):
        return min(math.floor((velocity / 128.0) * self.velocity_bins), self.velocity_bins - 1)
    def _dequantize_velocity(self, bin_idx):
        return int((bin_idx + 0.5) * (128.0 / self.velocity_bins))
    def _quantize_duration(self, duration_ticks, ticks_per_quarter):
        position_ticks = ticks_per_quarter / self.position_resolution
        duration_positions = max(1, round(duration_ticks / position_ticks))
        return min(duration_positions, self.duration_bins)
    def _dequantize_duration(self, bin_idx, ticks_per_quarter):
        position_ticks = ticks_per_quarter / self.position_resolution
        return int(bin_idx * position_ticks)
    def midi_to_tokens(self, midi_path):
        midi = miditoolkit.midi.parser.MidiFile(midi_path)
        ticks_per_quarter = midi.ticks_per_beat
        ticks_per_bar = ticks_per_quarter * 4
        notes = []
        for inst in midi.instruments:
            notes.extend(inst.notes)
        notes.sort(key=lambda x: (x.start, x.pitch))
        tokens = []
        current_bar = -1
        for note in notes:
            if not (21 <= note.pitch <= 108):
                continue
            bar = note.start // ticks_per_bar
            if bar > current_bar:
                for b in range(current_bar + 1, bar + 1):
                    tokens.append("Bar")
                current_bar = bar
            tick_in_bar = note.start % ticks_per_bar
            position = round(tick_in_bar / (ticks_per_quarter / self.position_resolution))
            position = min(max(position, 0), self.position_resolution - 1)
            pitch = note.pitch
            velocity = self._quantize_velocity(note.velocity)
            duration = self._quantize_duration(note.end - note.start, ticks_per_quarter)
            tokens.append(f"Position_{position}")
            tokens.append(f"Pitch_{pitch}")
            tokens.append(f"Velocity_{velocity}")
            tokens.append(f"Duration_{duration}")
        return tokens
    def tokens_to_midi(self, tokens, output_path):
        midi = miditoolkit.midi.parser.MidiFile()
        ticks_per_quarter = midi.ticks_per_beat
        ticks_per_bar = ticks_per_quarter * 4
        inst = miditoolkit.midi.containers.Instrument(program=0, is_drum=False, name="Piano")
        current_bar = -1
        current_position = 0
        current_pitch = -1
        current_velocity = -1
        i = 0
        while i < len(tokens):
            token = tokens[i]
            if token == "Bar":
                current_bar += 1
            elif token.startswith("Position_"):
                current_position = int(token.split("_")[1])
            elif token.startswith("Pitch_"):
                current_pitch = int(token.split("_")[1])
            elif token.startswith("Velocity_"):
                current_velocity = int(token.split("_")[1])
            elif token.startswith("Duration_"):
                duration_bin = int(token.split("_")[1])
                if current_bar >= 0 and current_pitch != -1 and current_velocity != -1:
                    start_tick = current_bar * ticks_per_bar + int(current_position * (ticks_per_quarter / self.position_resolution))
                    duration_ticks = self._dequantize_duration(duration_bin, ticks_per_quarter)
                    end_tick = start_tick + duration_ticks
                    velocity = self._dequantize_velocity(current_velocity)
                    note = miditoolkit.midi.containers.Note(
                        velocity=velocity,
                        pitch=current_pitch,
                        start=start_tick,
                        end=end_tick
                    )
                    inst.notes.append(note)
                current_pitch = -1
                current_velocity = -1
            i += 1
        midi.instruments.append(inst)
        midi.dump(output_path)


## 2. Data Preprocessing
This script pre-parses all MIDI files into PyTorch Tensors (`.pt`) to save CPU time and prevent Memory errors during training.

In [ ]:
import os
import pandas as pd
import torch
from tqdm import tqdm
from remi_tokenizer import REMITokenizer
def preprocess_maestro(csv_path, base_path, output_dir):
    """
    Reads all MIDI files in MAESTRO, converts them to token IDs, 
    and saves them as PyTorch tensors (.pt files).
    """
    os.makedirs(output_dir, exist_ok=True)
    tokenizer = REMITokenizer()
    df = pd.read_csv(csv_path)
    print(f"Starting preprocessing of {len(df)} files...")
    is_batch_run = os.environ.get('KAGGLE_KERNEL_RUN_TYPE', '') == 'Batch'
    for idx, row in tqdm(df.iterrows(), total=len(df), disable=is_batch_run):
        midi_filename = row['midi_filename']
        split = row['split']
        if split not in ['train', 'validation']:
            continue
        midi_path = os.path.join(base_path, midi_filename)
        save_sub_dir = os.path.join(output_dir, os.path.dirname(midi_filename))
        os.makedirs(save_sub_dir, exist_ok=True)
        save_name = os.path.basename(midi_filename).replace('.midi', '.pt').replace('.mid', '.pt')
        save_path = os.path.join(save_sub_dir, save_name)
        if os.path.exists(save_path):
            continue
        try:
            tokens = tokenizer.midi_to_tokens(midi_path)
            ids = tokenizer.vocab.encode(tokens)
            tensor_ids = torch.tensor(ids, dtype=torch.int16)
            torch.save(tensor_ids, save_path)
        except Exception as e:
            print(f"Error processing {midi_path}: {e}")
if __name__ == "__main__":
    csv_path = '/kaggle/input/datasets/jackvial/themaestrodatasetv2/maestro-v2.0.0/maestro-v2.0.0.csv'
    base_path = '/kaggle/input/datasets/jackvial/themaestrodatasetv2/maestro-v2.0.0/'
    output_dir = '/kaggle/working/processed_maestro'
    if os.path.exists(csv_path):
        preprocess_maestro(csv_path, base_path, output_dir)
    else:
        print("CSV not found. This script should be run in the Kaggle environment.")


## 3. Dataset and DataLoader
This module loads the preprocessed `.pt` files and implements a sliding window for sequence modeling.

In [ ]:
import os
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from remi_tokenizer import REMITokenizer
class MaestroDataset(Dataset):
    def __init__(self, csv_path, processed_dir, split="train", seq_len=1024):
        """
        Args:
            csv_path (str): Path to maestro-v2.0.0.csv
            processed_dir (str): Base path for preprocessed .pt files (/kaggle/working/processed_maestro/)
            split (str): 'train' or 'validation'
            seq_len (int): Sequence length for sliding window
        """
        self.processed_dir = processed_dir
        self.seq_len = seq_len
        df = pd.read_csv(csv_path)
        df = df[df['split'] == split].reset_index(drop=True)
        self.pt_files = []
        for fname in df['midi_filename']:
            pt_name = fname.replace('.midi', '.pt').replace('.mid', '.pt')
            pt_path = os.path.join(processed_dir, pt_name)
            self.pt_files.append(pt_path)
        self.token_cache = {}
        self.estimated_sequences_per_file = 20
    def __len__(self):
        return len(self.pt_files) * self.estimated_sequences_per_file
    def _get_tokens(self, file_idx):
        if file_idx not in self.token_cache:
            pt_path = self.pt_files[file_idx]
            try:
                tokens = torch.load(pt_path).long()
                self.token_cache[file_idx] = tokens
            except Exception as e:
                self.token_cache[file_idx] = torch.tensor([], dtype=torch.long)
        return self.token_cache[file_idx]
    def __getitem__(self, idx):
        attempts = 0
        max_attempts = 100
        while attempts < max_attempts:
            file_idx = (idx // self.estimated_sequences_per_file) % len(self.pt_files)
            token_ids = self._get_tokens(file_idx)
            if len(token_ids) >= self.seq_len + 1:
                break
            idx = torch.randint(0, len(self), (1,)).item()
            attempts += 1
        if attempts == max_attempts:
            raise RuntimeError(f"Could not find a valid sequence after {max_attempts} attempts. " 
                               f"Please ensure dataset is preprocessed correctly and files are larger than seq_len={self.seq_len}.")
        max_start = len(token_ids) - (self.seq_len + 1)
        start_idx = torch.randint(0, max_start + 1, (1,)).item()
        window = token_ids[start_idx : start_idx + self.seq_len + 1]
        input_ids = window[:-1]
        target_ids = window[1:]
        return input_ids, target_ids
def get_dataloader(csv_path, processed_dir, split="train", batch_size=8, seq_len=1024, num_workers=2):
    dataset = MaestroDataset(csv_path, processed_dir, split, seq_len)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=(split=="train"), 
                            num_workers=num_workers, drop_last=True)
    return dataloader


## 4. Decoder-Only Transformer Architecture
This defines the GPT-style Transformer using PyTorch's `TransformerEncoderLayer` with a causal mask, and a standard GPT LayerNorm and weight initialization.

In [ ]:
import math
import torch
import torch.nn as nn
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super(PositionalEncoding, self).__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)
    def forward(self, x):
        """
        Args:
            x: Tensor, shape [batch_size, seq_len, embedding_dim]
        """
        x = x + self.pe[:, :x.size(1)]
        return x
def generate_square_subsequent_mask(sz):
    """
    Generates an upper-triangular matrix of -inf, with zeros on diag.
    """
    mask = (torch.triu(torch.ones(sz, sz)) == 1).transpose(0, 1)
    mask = mask.float().masked_fill(mask == 0, float('-inf')).masked_fill(mask == 1, float(0.0))
    return mask
class MusicTransformer(nn.Module):
    def __init__(self, vocab_size, d_model=512, n_heads=8, num_layers=6, dim_feedforward=2048, dropout=0.1, max_seq_len=2048):
        super(MusicTransformer, self).__init__()
        self.d_model = d_model
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_encoder = PositionalEncoding(d_model, max_len=max_seq_len)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, 
            nhead=n_heads, 
            dim_feedforward=dim_feedforward, 
            dropout=dropout,
            batch_first=True
        )
        self.transformer_decoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.ln_f = nn.LayerNorm(d_model)
        self.output_head = nn.Linear(d_model, vocab_size)
        self.init_weights()
    def init_weights(self):
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.normal_(p, mean=0.0, std=0.02)
        self.output_head.bias.data.zero_()
    def forward(self, src, src_mask=None):
        """
        Args:
            src: Tensor, shape [batch_size, seq_len]
            src_mask: Tensor, causal mask
        Returns:
            output: Tensor, shape [batch_size, seq_len, vocab_size]
        """
        src = self.embedding(src) * math.sqrt(self.d_model)
        src = self.pos_encoder(src)
        if src_mask is None:
            device = src.device
            sz = src.size(1)
            src_mask = generate_square_subsequent_mask(sz).to(device)
        output = self.transformer_decoder(src, mask=src_mask, is_causal=True)
        output = self.ln_f(output)
        logits = self.output_head(output)
        return logits


## 5. Training Loop
This module handles the training loop with Automatic Mixed Precision (AMP), gradient scaling, checkpointing, and a Cosine Annealing Learning Rate Scheduler.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.cuda.amp import autocast, GradScaler
from tqdm import tqdm
import os
def train_epoch(model, dataloader, optimizer, criterion, scaler, scheduler, device, epoch):
    model.train()
    total_loss = 0.0
    is_batch_run = os.environ.get('KAGGLE_KERNEL_RUN_TYPE', '') == 'Batch'
    pbar = tqdm(dataloader, desc=f"Epoch {epoch} [Train]", disable=is_batch_run)
    for batch_idx, (input_ids, target_ids) in enumerate(pbar):
        input_ids, target_ids = input_ids.to(device), target_ids.to(device)
        optimizer.zero_grad(set_to_none=True)
        with autocast():
            logits = model(input_ids)
            logits = logits.view(-1, logits.size(-1))
            target_ids = target_ids.view(-1)
            loss = criterion(logits, target_ids)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        total_loss += loss.item()
        pbar.set_postfix({'loss': loss.item(), 'lr': f"{scheduler.get_last_lr()[0]:.2e}"})
    avg_loss = total_loss / len(dataloader)
    return avg_loss
def evaluate(model, dataloader, criterion, device, epoch):
    model.eval()
    total_loss = 0.0
    is_batch_run = os.environ.get('KAGGLE_KERNEL_RUN_TYPE', '') == 'Batch'
    pbar = tqdm(dataloader, desc=f"Epoch {epoch} [Val]", disable=is_batch_run)
    with torch.no_grad():
        for input_ids, target_ids in pbar:
            input_ids, target_ids = input_ids.to(device), target_ids.to(device)
            with autocast():
                logits = model(input_ids)
                logits = logits.view(-1, logits.size(-1))
                target_ids = target_ids.view(-1)
                loss = criterion(logits, target_ids)
            total_loss += loss.item()
            pbar.set_postfix({'val_loss': loss.item()})
    avg_loss = total_loss / len(dataloader)
    return avg_loss
def train_model(model, train_loader, val_loader, vocab, epochs=10, lr=5e-4, save_dir='/kaggle/working/'):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = model.to(device)
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    total_steps = len(train_loader) * epochs
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=total_steps, eta_min=1e-6)
    criterion = nn.CrossEntropyLoss(ignore_index=vocab.pad_id)
    scaler = GradScaler()
    best_val_loss = float('inf')
    os.makedirs(save_dir, exist_ok=True)
    print(f"Training started on {device}")
    for epoch in range(1, epochs + 1):
        train_loss = train_epoch(model, train_loader, optimizer, criterion, scaler, scheduler, device, epoch)
        val_loss = evaluate(model, val_loader, criterion, device, epoch)
        print(f"Epoch {epoch} Summary: Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            save_path = os.path.join(save_dir, "best_music_transformer.pt")
            torch.save(model.state_dict(), save_path)
            print(f"--> Saved new best model to {save_path} (Val Loss: {best_val_loss:.4f})")


## 6. Autoregressive Generation
This script handles the inference using Top-K and Top-P (Nucleus) sampling to generate creative sequences autoregressively.

In [ ]:
import torch
import torch.nn.functional as F
from tqdm import tqdm
import os
def top_k_top_p_filtering(logits, top_k=0, top_p=0.0, filter_value=-float('Inf')):
    """ Filter a distribution of logits using top-k and/or nucleus (top-p) filtering """
    if top_k > 0:
        indices_to_remove = logits < torch.topk(logits, top_k)[0][..., -1, None]
        logits[indices_to_remove] = filter_value
    if top_p > 0.0:
        sorted_logits, sorted_indices = torch.sort(logits, descending=True)
        cumulative_probs = torch.cumsum(F.softmax(sorted_logits, dim=-1), dim=-1)
        sorted_indices_to_remove = cumulative_probs > top_p
        sorted_indices_to_remove[..., 1:] = sorted_indices_to_remove[..., :-1].clone()
        sorted_indices_to_remove[..., 0] = 0
        indices_to_remove = sorted_indices_to_remove.scatter(dim=1, index=sorted_indices, src=sorted_indices_to_remove)
        logits[indices_to_remove] = filter_value
    return logits
def generate_music(model, tokenizer, prompt_tokens=["Bar"], max_length=1024, temperature=1.0, top_k=50, top_p=0.9, save_path="/kaggle/working/generated.mid"):
    """
    Autoregressively generate music sequence.
    """
    device = next(model.parameters()).device
    model.eval()
    input_ids = tokenizer.vocab.encode(prompt_tokens)
    input_tensor = torch.tensor([input_ids], dtype=torch.long).to(device)
    print(f"Starting generation for {max_length} tokens...")
    with torch.no_grad():
        is_batch_run = os.environ.get('KAGGLE_KERNEL_RUN_TYPE', '') == 'Batch'
        for _ in tqdm(range(max_length), disable=is_batch_run):
            logits = model(input_tensor)
            next_token_logits = logits[0, -1, :] / (temperature if temperature > 0 else 1.0)
            filtered_logits = top_k_top_p_filtering(next_token_logits.unsqueeze(0), top_k=top_k, top_p=top_p)
            probabilities = F.softmax(filtered_logits, dim=-1)
            next_token_id = torch.multinomial(probabilities, num_samples=1)
            input_tensor = torch.cat([input_tensor, next_token_id], dim=1)
            if next_token_id.item() == tokenizer.vocab.eos_id:
                print("Generated <EOS> token. Stopping early.")
                break
    generated_ids = input_tensor[0].cpu().tolist()
    generated_tokens = tokenizer.vocab.decode(generated_ids)
    print(f"Saving generated MIDI to {save_path}")
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    tokenizer.tokens_to_midi(generated_tokens, save_path)
    return generated_tokens


## 7. Main Execution Block
This block ties everything together. It initializes the modules, starts the training loop, and runs a generation test.

In [ ]:
def main():
    csv_path = '/kaggle/input/datasets/jackvial/themaestrodatasetv2/maestro-v2.0.0/maestro-v2.0.0.csv'
    base_path = '/kaggle/input/datasets/jackvial/themaestrodatasetv2/maestro-v2.0.0/'
    processed_dir = '/kaggle/working/processed_maestro'
    if not os.path.exists(processed_dir) or len(os.listdir(processed_dir)) == 0:
        print("Preprocessing data...")
        preprocess_maestro(csv_path, base_path, processed_dir)
    else:
        print("Data already preprocessed.")
    batch_size = 8
    seq_len = 1024
    d_model = 512
    n_heads = 8
    num_layers = 6
    epochs = 10
    tokenizer = REMITokenizer()
    vocab_size = len(tokenizer.vocab)
    print(f"Vocabulary Size: {vocab_size}")
    print("Initializing DataLoaders...")
    train_loader = get_dataloader(csv_path, processed_dir, split="train", batch_size=batch_size, seq_len=seq_len, num_workers=2)
    val_loader = get_dataloader(csv_path, processed_dir, split="validation", batch_size=batch_size, seq_len=seq_len, num_workers=2)
    print("Initializing Model...")
    model = MusicTransformer(
        vocab_size=vocab_size,
        d_model=d_model,
        n_heads=n_heads,
        num_layers=num_layers,
        max_seq_len=seq_len + 1
    )
    train_model(model, train_loader, val_loader, tokenizer.vocab, epochs=epochs, lr=5e-4)
    best_model_path = '/kaggle/working/best_music_transformer.pt'
    if os.path.exists(best_model_path):
        model.load_state_dict(torch.load(best_model_path))
        print("Loaded best model for generation.")
    print("Generating sample music...")
    generate_music(
        model, 
        tokenizer, 
        prompt_tokens=["Bar", "Position_0", "Pitch_60", "Velocity_16", "Duration_16"], 
        max_length=512, 
        temperature=1.0, 
        save_path='/kaggle/working/generated.mid'
    )
if __name__ == "__main__":
    if os.path.exists('/kaggle/input/datasets/jackvial/themaestrodatasetv2/maestro-v2.0.0/maestro-v2.0.0.csv'):
        main()
    else:
        print("Dataset not found. Please ensure it is attached to the Kaggle notebook.")


## Convert generated MIDI to MP3
Run this cell to convert the generated `.mid` file into an `.mp3` file for easy playback.

In [ ]:
!fluidsynth -ni /usr/share/sounds/sf2/FluidR3_GM.sf2 /kaggle/working/generated.mid -F /kaggle/working/generated.wav -r 44100
!ffmpeg -y -i /kaggle/working/generated.wav -b:a 192k /kaggle/working/generated.mp3
from IPython.display import Audio
Audio('/kaggle/working/generated.mp3')